# 课程13 - 带有Cognee知识图谱的代理记忆


## 设置

本笔记本演示如何使用 [**Cognee**](https://www.cognee.ai/) 知识图谱和 **Microsoft Agent Framework** (MAF) 构建具有持久内存的智能<strong>编码助手</strong>。

Cognee 将非结构化文本转化为结构化、可查询的知识图谱，背后由向量嵌入支持——为您的代理提供丰富的、关系感知的长期记忆。

### 您将学习到
1. <strong>构建知识图谱</strong>: 将开发者档案和最佳实践转化为结构化、可查询的知识。
2. **集成 Cognee 与 MAF**: 使用 `@tool` 函数让 MAF 代理查询 Cognee 的知识图谱。
3. <strong>会话感知对话</strong>: 维护同一会话中多个问题的上下文。
4. <strong>长期记忆</strong>: 在会话之间持久保存重要知识，并在新对话中检索。

### 先决条件
- Python 3.9+
- 本地运行的 Redis (`docker run -d -p 6379:6379 redis`) 用于会话管理
- 一个 LLM API 密钥（例如 OpenAI）——在 `.env` 中设置 `LLM_API_KEY`
- `.env` 中设置 `CACHING=true`（Cognee 会话所需）
- 一个部署了聊天模型的 Microsoft Foundry 项目
- `.env` 中设置 `AZURE_AI_PROJECT_ENDPOINT` 和 `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- 已通过 Azure CLI 验证身份（`az login`）


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")


In [ ]:
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ FoundryChatClient created")


## 代理记忆类型

本笔记本探讨了主课程第13课笔记本中的相同三种记忆类型，但使用 Cognee 作为长期记忆后端：

| 记忆类型 | 机制 | 生命周期 |
|---|---|---|
| <strong>工作记忆</strong> | `agent.create_session()` (MAF) | 单次对话 |
| <strong>短期记忆</strong> | Cognee 会话缓存 (Redis) | 单次会话 |
| <strong>长期记忆</strong> | Cognee 知识图谱 + 向量 | 永久 |

### Cognee 的记忆架构
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## 准备 Cognee 存储


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## 第一部分 — 构建知识库

我们摄取三种类型的数据，以创建一个全面的编程助手知识库：

1. <strong>开发者个人资料</strong> — 个人专业知识和技术背景
2. **Python最佳实践** — Python之禅及实用指南
3. <strong>历史对话</strong> — 开发者与AI助手之间的过去问答记录


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## 可视化知识图谱

Cognee 可以渲染它提取的实体和关系的交互式 HTML 可视化。


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## 用 Memify 丰富记忆

`memify()` 分析知识图并生成智能规则 —— 识别模式、最佳实践以及概念之间的关系。


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## 第二部分 — 使用 Cognee 工具的 MAF 代理

现在我们创建一个可以通过 `@tool` 函数查询 Cognee 知识图谱的 MAF 代理。这样代理就能利用图结构感知的语义搜索的全部优势，同时通过会话保持对话上下文。


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = provider.as_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")


## 使用会话的工作记忆

`AgentSession`（通过 `agent.create_session()` 创建）在会话中提供工作记忆。代理可以回顾之前的消息，同时查询 Cognee 的长期知识图谱。


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## 新会话 — 长期记忆持续存在

开始一个新的会话会清除工作记忆，但Cognee知识图谱仍然可用。代理可以在一个全新的对话中检索相同的长期知识。


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## 摘要

在本笔记本中，您构建了一个结合了<strong>MAF工作记忆</strong>（`agent.create_session()`）与<strong>Cognee长期知识图谱</strong>的编码助手。

### 您学到了什么
1. <strong>知识图谱构建</strong>：Cognee摄取非结构化文本并构建图谱 + 向量记忆。
2. **利用memify丰富图谱**：在现有图谱基础上派生事实和更丰富的关系。
3. **MAF + Cognee集成**：`@tool`函数让MAF代理自然查询Cognee的图谱。
4. **工作记忆 + 长期记忆**：`AgentSession`（通过`agent.create_session()`）提供会话上下文，Cognee提供持久知识。
5. **使用NodeSets的过滤搜索**：定位知识图中特定子集（例如，仅原则）。

### 主要收获
- **Cognee** 将原始文本转化为结构化、具备关系意识的记忆 —— 比单纯的向量存储更强大。
- **`@tool`函数** 清晰桥接MAF代理与外部知识系统。
- **`AgentSession`**（通过`agent.create_session()`）使每个对话上下文与长期知识分离保存。
- 相同的知识图谱服务于多个会话和代理。

### 现实应用
- <strong>开发者助手</strong>：代码审查、事件分析、架构助理
- <strong>面向客户的助手</strong>：基于产品文档、FAQ及CRM记录的支持代理
- <strong>内部专家助手</strong>：基于政策、法律或安全指南的推理助手
- <strong>统一数据层</strong>：将结构化与非结构化数据合成一个可查询图谱

### 后续步骤
- 在Cognee中尝试时间感知功能
- 为特定领域图谱质量定义OWL本体
- 添加用户反馈机制，提升检索效果
- 扩展到共享同一Cognee记忆层的多代理系统


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
